In [1]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

In [3]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03 - training/experiment_tracking/mlruns/1', creation_time=1775520215694, experiment_id='1', last_update_time=1775520215694, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/03 - training/experiment_tracking/mlruns/0', creation_time=1775517629048, experiment_id='0', last_update_time=1775517629048, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [4]:
client.create_experiment(name="my-cool-experiment")

'2'

In [29]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [31]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 905203bbcf2f4c20881d74de289c6b7e, rmse: 6.3099
run id: 8469035ef211468799bb5689f4378c2e, rmse: 6.3100
run id: 74232f705a104818ad342593d14c9777, rmse: 6.3109
run id: abcc9d5e6b4741d78d93e342f61569e0, rmse: 6.3120
run id: 04b2280d7f074db993c76a3d120bbc60, rmse: 6.3144


In [32]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [36]:
run_id = "8469035ef211468799bb5689f4378c2e"
model_uri = "http://127.0.0.1:5000/#/experiments/1/models/m-5b3044b28b6040ec86956024267ecadb"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '2' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1775606418190, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1775606418190, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id=None, run_link=None, source='http://127.0.0.1:5000/#/experiments/1/models/m-5b3044b28b6040ec86956024267ecadb', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [54]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Staging
version: 2, stage: Production


/tmp/ipykernel_27366/669935608.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [52]:
model_version = 2
new_stage = "Production"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_27366/3059153048.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=['Staging'], creation_timestamp=1775606418190, current_stage='Production', deployment_job_state=None, description='The model version 2 was transitioned to Staging on 2026-04-08', last_updated_timestamp=1775606709631, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id=None, run_link=None, source='http://127.0.0.1:5000/#/experiments/1/models/m-5b3044b28b6040ec86956024267ecadb', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [40]:
client.set_registered_model_alias(
    name=model_name,
    version=2,
    alias=new_stage
)


In [41]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=['Staging'], creation_timestamp=1775606418190, current_stage='Staging', deployment_job_state=None, description='The model version 2 was transitioned to Staging on 2026-04-08', last_updated_timestamp=1775606462786, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id=None, run_link=None, source='http://127.0.0.1:5000/#/experiments/1/models/m-5b3044b28b6040ec86956024267ecadb', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

In [ ]:
from sklearn.metrics import mean_squared_error
import pandas as pd

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}


In [44]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/mlops-zoomcamp/03 - training/experiment_tracking/preprocessor'

In [47]:
import pickle

with open("preprocessor/preprocessor.bin", "rb") as f_in:
    dv = pickle.load(f_in)

In [49]:
X_test = preprocess(df, dv)

In [50]:
target = "duration"
y_test = df[target].values

In [ ]:
model_name

'nyc-taxi-regressor'

In [63]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: user 20.2 ms, sys: 4.11 ms, total: 24.3 ms
Wall time: 124 ms


ValueError: not enough values to unpack (expected 2, got 1)